# Medium GEXF demo: SiS Words

This dataset includes GEXF viz encodings for node color, size, and position.
The source uses a single color and size value, so the default plot looks uniform.
Below we show the faithful default binding, how to drop GEXF colors/sizes while
keeping layout, and then how to apply Graphistry encodings.


In [1]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import graphistry


In [2]:
GRAPHISTRY_SERVER = os.environ.get("GRAPHISTRY_SERVER", "hub.graphistry.com")
GRAPHISTRY_PROTOCOL = os.environ.get("GRAPHISTRY_PROTOCOL", "https")
GRAPHISTRY_USERNAME = os.environ.get("GRAPHISTRY_USERNAME")
GRAPHISTRY_PASSWORD = os.environ.get("GRAPHISTRY_PASSWORD")

if not GRAPHISTRY_USERNAME or not GRAPHISTRY_PASSWORD:
    raise RuntimeError("Set GRAPHISTRY_USERNAME and GRAPHISTRY_PASSWORD to upload.")

graphistry.register(
    api=3,
    protocol=GRAPHISTRY_PROTOCOL,
    server=GRAPHISTRY_SERVER,
    username=GRAPHISTRY_USERNAME,
    password=GRAPHISTRY_PASSWORD,
)


In [3]:
DATA_URL = "https://raw.githubusercontent.com/medialab/medialab-network-dataset/master/SiS%20Words.gexf"
DATA_DIR = Path("data")
GEXF_PATH = DATA_DIR / "sis_words.gexf"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not GEXF_PATH.exists():
    urlretrieve(DATA_URL, GEXF_PATH)

GEXF_PATH.exists()


True

In [4]:
g = graphistry.gexf(str(GEXF_PATH))
counts = {"nodes": len(g._nodes), "edges": len(g._edges)}
bindings = {
    "point_color": g._point_color,
    "point_size": g._point_size,
    "point_x": g._point_x,
    "point_y": g._point_y,
    "edge_color": g._edge_color,
    "play": g._url_params.get("play"),
}
counts, bindings


({'nodes': 6704, 'edges': 71744},
 {'point_color': 'viz_color',
  'point_size': 'viz_size',
  'point_x': 'viz_x',
  'point_y': 'viz_y',
  'edge_color': None,
  'play': 0})

In [5]:
g._nodes.head()


,node_id,label,class,main,occurences,viz_size,viz_x,viz_y,viz_z,viz_color
0,w70401,populations indigènes,populations et amélioration des conditions de vie,True,3,10.0,-649.47797,-996.466860,0.0,#999999
1,w70416,impact des activités humaines,réchauffement climatique et elavation du nivea...,True,2,10.0,789.25270,10.201024,0.0,#999999
2,w70453,préservation de la qualité,développement durable et environnement,True,3,10.0,1131.04210,-927.317500,0.0,#999999
3,w70455,préservation de la nature,préservation de la nature et de la biodiversité,True,2,10.0,1068.51270,-995.344000,0.0,#999999
4,w70454,préservation des ressources naturelles,développement durable et environnement,True,4,10.0,982.29270,-890.796300,0.0,#999999


In [6]:
g._edges.head()


,source,target
0,w70401,w69745
1,w70401,w69741
2,w70401,w54632
3,w70401,w53692
4,w70401,w53637


In [7]:
g.name("SiS Words (GEXF defaults)").plot(render="url")


'https://hub.graphistry.com/graph/graph.html?dataset=76d6cd0489f64caf820747e4593862d3&type=arrow&viztoken=7df9f3fa-841b-4fcc-a57f-c813f49416a2&usertag=ef9e6f8d-pygraphistry-0.50.5+4.g840b4ea3.dirty&splashAfter=1769485277&info=true&play=0'

## Drop GEXF colors/sizes (keep layout)

Use `bind_node_viz` / `bind_edge_viz` to keep only the bindings you want.
Here we keep position for layout, and drop color/size/opacity/icon bindings.


In [8]:
g_layout_only = graphistry.gexf(
    str(GEXF_PATH),
    bind_node_viz=["position"],
    bind_edge_viz=[],
)


In [9]:
g_layout_only.name("SiS Words (layout only)").plot(render="url")


'https://hub.graphistry.com/graph/graph.html?dataset=9996568466c24a99b85582673c9ef0a7&type=arrow&viztoken=f3b2946f-fa8b-40f4-a6ba-c1c0e021c6d2&usertag=ef9e6f8d-pygraphistry-0.50.5+4.g840b4ea3.dirty&splashAfter=1769485293&info=true&play=0'

## Apply Graphistry encodings

After dropping bindings, use Graphistry encodings for color/size.
Here we color by `class` and size by `occurences`.


In [10]:
required_cols = ["class", "occurences"]
missing_cols = [col for col in required_cols if col not in g_layout_only._nodes.columns]
assert not missing_cols, f"Missing expected node columns: {missing_cols}"

g_encoded = (
    g_layout_only
    .encode_point_color(
        "class",
        palette=["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#EECA3B", "#B279A2", "#FF9DA6"],
        as_categorical=True,
    )
    .encode_point_size("occurences")
)


In [11]:
g_encoded.name("SiS Words (layout + encodings)").plot(render="url")


'https://hub.graphistry.com/graph/graph.html?dataset=db31bed1e0304487acd72abea1079876&type=arrow&viztoken=fe1159e9-6572-4845-8a27-201cf6aaae90&usertag=ef9e6f8d-pygraphistry-0.50.5+4.g840b4ea3.dirty&splashAfter=1769485310&info=true&play=0'